# START HERE — faster, lower-reasoning pilot

1. **Reasoning settings: minimal versus low**, replacing the earlier low versus high comparison. Same GPT-5 nano model; both new settings are intended to use less reasoning. Actual savings must be measured.
**Prompt audit update:** history now comes first and the authoritative current state last. Observation fields are copied before the order; zeros must stay zero. Rules questions name their output fields. This keeps real correctness checks rather than forcing their answers in the schema. The final report includes first-attempt reliability, so retries cannot hide errors. Live improvement remains unverified.

**Completion target:** `groups_per_cell=1` means one fully completed four-role group in each of four conditions. After three incorrect attempts, abandon that entire group and try a fresh one, up to three replacements per target group. Keep collecting until the target is reached or the $0.75 guard/replacement limit stops the run. Never replace a group merely because beta is unusable or an effect is not significant.

2. **Eight concurrent requests**, four groups × four roles × 36 weeks: 16 agents, 576 game decisions, plus 20 checks = **596 planned requests**. This is a full-game measurement pilot, not a few-request connection test.
3. **$0.75 local spending allowance** stays in place. No automatic budget increase, model switch, or larger experiment. Incorrect observation/rules replies get at most two logged retries. Parallelism saves waiting; lower reasoning effort is the change intended to save tokens.
4. Both settings share an **8,192-token output ceiling**, including reasoning and the answer. This is a maximum, not a target. Reducing effort is the main speed control; exhausting the ceiling still stops the run and preserves the failure.
5. Standard cost Beer Game: 0.50 holding + 1.00 backlog per unit/week, no production charge or sales revenue, initial stock 12 and flow 4. External demand is 4 for four weeks, then 8. Agents choose orders from week 1.
6. Reports show supply-chain performance, usable β, actual reasoning tokens and latency. **One group per condition does not establish H1/H2 statistically.** H2_API now compares minimal versus low reasoning; it does not measure 60/120-second human time pressure. Less separation may make the effect harder to detect. We do not choose settings to obtain significance.

**Run:** if an old cell is still running, interrupt it first. Reopen this notebook from disk, then Restart Kernel → Run All → enter your key in the hidden popup. This creates a new run and preserves earlier results. Run the final report cell after completion or a stop. Do not mix previous low/high data into this revised comparison.

**Time and cost:** no live minimal/low full game has been measured yet. The previous low-effort startup calls averaged about 6 seconds, but later histories can take longer. If requests average 3–8 seconds, 596 requests / 8 concurrent gives an idealized 4–10 minutes; weekly barriers, uneven requests and rate limits add time. Treat this as arithmetic, not a promised runtime. If average input/output usage is 2,000/500 tokens, estimated cost is about $0.18; at 4,000/1,000 it is $0.36. These are scenarios, not measured forecasts. The progress bar and final report provide actual measurements.

For a free software check, set `MODE = "offline"`. Synthetic results do not verify model behavior or API savings.


In [6]:
# Uncomment if dependencies are missing, then restart the kernel.
# %pip install "openai>=2,<3" "pydantic>=2.8,<3" numpy pandas scipy matplotlib ipykernel
from pathlib import Path
import sys, importlib, os, getpass
from IPython.display import display, Markdown, Image
BASE = Path.cwd()
if (BASE / "Project" / "beer_game_experiment").is_dir():
    BASE = BASE / "Project" / "beer_game_experiment"
elif (BASE / "beer_game_experiment").is_dir():
    BASE = BASE / "beer_game_experiment"
if not (BASE / "budget_verification.py").exists():
    raise FileNotFoundError("Open this notebook beside budget_verification.py in beer_game_experiment.")
sys.path.insert(0, str(BASE))
import budget_verification as study
importlib.reload(study)


<module 'budget_verification' from '/Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/budget_verification.py'>

## 1. Settings

The API uses actual `minimal` and `low` effort names throughout the data and reports. Both share the same 8,192-token ceiling and 300-second infrastructure timeout; that timeout is not a requested thinking duration. The model can finish earlier and is billed for actual usage. Lower reasoning effort can change decision quality; state readbacks and β identification checks remain required.

[Official OpenAI documentation: GPT-5 reasoning settings](https://developers.openai.com/api/docs/guides/latest-model?model=gpt-5) · [Model pricing](https://developers.openai.com/api/docs/models/gpt-5-nano)


In [7]:
MODE = "api"                    # Use "offline" for a free software check
BUDGET_USD = 0.75                # Local allowance across this entire verification run
REPLICATE_ID = "cheap-model-prompt-audit-08"
MAX_CONCURRENT_REQUESTS = 8      # Shared limit across all groups and roles
CONFIG = dict(mode=MODE, stage="game", model=study.MODEL,
              groups_per_cell=1, weeks=36, replicate_id=REPLICATE_ID,
              seed=20260927, budget_usd=BUDGET_USD,
              max_output_tokens=8192, timeout_s=300,
              max_concurrent_requests=MAX_CONCURRENT_REQUESTS,
              effort_levels=study.EFFORT_LEVELS, max_attempts=3, max_group_replacements=3)
print("Model:", study.MODEL)
print("Concurrent requests:", MAX_CONCURRENT_REQUESTS)
print("Reasoning comparison:", " versus ".join(study.EFFORT_LEVELS))
print("Lower effort should save time/tokens; actual runtime and cost remain unmeasured.")
print("Mode:", MODE, "| 4 groups / 16 agents / 36 weeks / 596 planned responses")
print(f"Local allowance: ${BUDGET_USD:.2f}; no automatic model upgrade or increase in allowance.")
print("Keep settings unchanged to reuse successful cached requests. Only raising the allowance resumes a spending stop.")


Model: gpt-5-nano-2025-08-07
Concurrent requests: 8
Reasoning comparison: minimal versus low
Lower effort should save time/tokens; actual runtime and cost remain unmeasured.
Mode: api | 4 groups / 16 agents / 36 weeks / 596 planned responses
Local allowance: $0.75; no automatic model upgrade or increase in allowance.
Keep settings unchanged to reuse successful cached requests. Only raising the allowance resumes a spending stop.


## 2. Free software verification

Known-answer β recovery and a demand-copying counterexample check the analysis code before paid requests. Passing is not evidence that GPT will follow the assumed ordering model. We judge the real pilot by measurement quality, never by whether the training effect has the desired sign.


In [8]:
checks = study.measurement_self_check()
display(checks)
assert checks.passed.all(), "Estimator self-check failed; do not proceed to paid requests."


,case,true_beta,estimated_beta,usable,passed
0,known beta 0.0,0.00,4.744894e-07,True,True
1,known beta 0.3,0.30,3.000000e-01,True,True
2,known beta 0.65,0.65,6.500000e-01,True,True
3,known beta 1.0,1.00,1.000000e+00,True,True
4,copy demand: beta must be unidentified,NaN,NaN,False,True


## 3. Run the verification — this is the only paid cell

The first four requests check actual ordering decisions under control/trained × minimal/low reasoning with structured output. Then four full games run. All roles use the same weekly snapshot before orders are committed. Incorrect observation/rules replies get up to two retries (three attempts total). The first valid reply is accepted; all attempts and costs remain saved. After exhausted correctness retries, the failed four-role group is saved and a fresh group starts in the same condition. Each target group allows up to three replacement groups. Refused/incomplete/API errors still stop the run. No invented orders are used. The progress bar includes access checks and rules checks.

If stopped, the next cell still reports what happened. Do not keep rerunning a failed request or spend the rest of your $3 without reviewing it. A spending stop can resume successful cached requests if you deliberately raise only BUDGET_USD; a saved failed response requires review.

Independent groups and same-week roles run concurrently, with at most 8 requests in flight. The bar shows completed/remaining requests, active requests, and each group's week. On a terminal failure, no new requests are sent; already-sent requests finish and save their usage before stopping. The total spending reservation includes every in-flight request. If interrupted, unknown usage stays reserved. Rate-limit errors stop the run without automatic retries.

Rules-check clarification: `total_holding_cost` explicitly means the charge for all units, not the per-unit rate. Each candidate group must pass its four rules checks before its game begins. The correct answer is not supplied to the model or substituted by code. Incorrect answers are logged with expected/received values and may be retried twice. This new protocol preserves the earlier failed pilot; clarification may help but does not guarantee minimal reasoning is reliable.

**Bounded retry policy:** three attempts maximum per logical request. Only incorrect observation/rules replies qualify; authentication, billing, refusal, token-limit and other API errors do not. Retries keep the model, reasoning effort, game state and history fixed, with a generic recheck reminder; they do not supply the correct answer or a preferred order. Every attempt counts against $0.75. Resuming the same run reuses saved attempts and cannot reset the retry limit. The progress total includes retries, and the report shows recovery counts by condition.


In [9]:
client = None
results = None
run_dir = study.latest_run(BASE, CONFIG)
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("An API key is required in API mode.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=CONFIG["timeout_s"])
        del key
    results, run_dir = await study.run_study(BASE, CONFIG, client=client)
    print("VERIFICATION COMPLETE. Run the next cell for the results.")
except study.ExperimentStopped as exc:
    print(str(exc))
    print("Run the next cell to review the saved status. Replacement limit, spending guard, or non-retryable error stopped the run. No automatic model upgrade.")
finally:
    if client is not None:
        await client.close()
    print("Results folder:", run_dir)


Stopped: api_error (RateLimitError, HTTP 429). See 89025eae4b1099be1e2d730b26065a2256771d009f6e85a30e6a2cb5df0d6375.json. No fabricated order or retry-until-correct was used. For HTTP 404 check model availability/access; for 429 check credits/rate limits; for 401 check the key; for incomplete responses review the output allowance. Preserve this failed pilot. Attempts used: 1/3.
Run the next cell to review the saved status. Replacement limit, spending guard, or non-retryable error stopped the run. No automatic model upgrade.
Results folder: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/results/verify_5nano_api_00e9f59be14e831d


## 4. Results — read these before paying for another run

The summary shows:

- Completed vs stopped, successful replies and failures, estimated spending.
- Supply-chain cost, customer fulfillment, inventory/backlog/order graphs.
- Usable β for each of the 16 roles and each complete group; blanks mean unidentified.
- Zero-order rate and agents that copied demand every week. Zero orders alone are not failures.
- Actual minimal/low reasoning-token use.
- Cost scenarios for larger GPT-5 nano runs and **repricing the same token counts** at GPT-5.4 nano rates. Repricing is not a prediction of another model's actual token use.

A promising screen means complete validated games, measurable β in all four groups, and descriptive effort separation. It does not establish hypotheses or justify selecting whichever model produces a preferred result. If it fails these checks, keep the results and review the cause before scaling. A different model still needs its own small verification.

Failed and replacement groups appear in `group_attempts.csv`; all committed candidate rounds remain in `all_candidate_rows.csv`. Main performance/beta tables use completed groups only. Counts of exclusions are reported. If replacement occurred, H1/H2 p-values are withheld because completing only selected responders can bias inference. Completion is not guaranteed within the allowance.

Read `first_attempt_reliability.csv` before scaling. Successful retries do not establish that the initial responses were reliable. Rules/readback checks do not prove strategic comprehension. See `PROMPT_AND_STABILITY_AUDIT.md` beside this notebook for findings, changes, and remaining limitations.


In [10]:
if (run_dir / "progress.json").exists():
    report = study.verification_report(run_dir, CONFIG)
    display(Markdown(report))
    figure = run_dir / "supply_chain_performance.png"
    if figure.exists():
        display(Image(filename=str(figure)))
    print("Report:", run_dir / "VERIFICATION_REPORT.md")
    print("Summary:", run_dir / "verification_summary.json")
    print("Tell me: 'the verification finished'. I can inspect these saved files.")
else:
    print("No run records yet. Complete the run cell or report the access/setup error.")


# Budget verification — conclusions

1. Real GPT API run.
2. Run stopped before completion: 307 successful responses; 5 saved failed/pending requests.
3. Ordering checks across conditions: 4/4 passed. Recorded estimated charges/reservations: $0.0601; allowance $0.75.
4. H1/H2 are not established by this pilot. One group per condition is for measurement verification only.

Retries: 0 failed attempts later recovered; 5 failed/pending attempts remain unresolved. All attempt costs are included.
The protocol permits at most two retries for incorrect observation/rules replies, using the same model and effort. No desired order, beta, or hypothesis result is supplied. Results describe agents with this retry policy; retries add resources and may differ by condition.

## First-attempt reliability

```
                 first_attempt_requests  first_attempt_valid  first_attempt_valid_rate
trained effort                                                                        
False   low                          53                   52                     0.981
        minimal                     109                  109                     1.000
True    low                          45                   41                     0.911
        minimal                     105                  105                     1.000
```
These checks measure rules answers and accurate copying of observations. Passing does not prove optimal ordering, understanding of delays, or identifiable beta. Retries can conceal a weak first-attempt rate, so inspect this table before scaling.

## Retry counts by condition

```
                 attempts  logical_requests  recovered_failed_attempts  extra_attempts
trained effort                                                                        
False   low            53                53                          0               0
        minimal       109               109                          0               0
True    low            45                45                          0               0
        minimal       105               105                          0               0
```

Group collection: 0/4 required groups completed; 0 failed groups retained in the audit.
Completed-group performance and beta exclude failed partial games. all_candidate_rows.csv retains committed rounds from all candidates; individual request files retain uncommitted replies. Replacement is based only on exhausted correctness failures, never beta or effect direction. Replacing failures selects for successful responders; confirmatory H1/H2 p-values are withheld if replacement occurred.

Do not interpret partial games as H1/H2 evidence. The last successful requests and any failure remain saved. If the local allowance stopped the run, review spending before raising it. If a request failed, keep the record and review the error; do not keep retrying blindly.

Standard scoring: 0.50 per held unit/week + 1.00 per backlogged unit/week; no sales revenue or production fee. Four roles, local information, 12 initial units per role; external demand 4 in weeks 1–4 and 8 afterward. Free choices start in week 1; this is a standard-cost implementation, not a claim to reproduce every classroom instruction. H2_API concerns reasoning effort; it cannot establish human time-pressure effects.

A useful verification result means complete responses, measurable beta, interpretable performance, and affordable token use. It does NOT mean the treatment effect has the predicted sign.


Report: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/results/verify_5nano_api_00e9f59be14e831d/VERIFICATION_REPORT.md
Summary: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/results/verify_5nano_api_00e9f59be14e831d/verification_summary.json
Tell me: 'the verification finished'. I can inspect these saved files.
